# Финальное задание по Dota 2

## Подготовка данных

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import datetime

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Чтение данных
features = pd.read_csv('./data/features.csv', index_col='match_id')
print('Размер таблицы:', features.shape)
features.head(3)

Размер таблицы: (97230, 108)


,start_time,lobby_type,r1_hero,r1_level,r1_xp,r1_gold,r1_lh,r1_kills,r1_deaths,r1_items,...,dire_boots_count,dire_ward_observer_count,dire_ward_sentry_count,dire_first_ward_time,duration,radiant_win,tower_status_radiant,tower_status_dire,barracks_status_radiant,barracks_status_dire
match_id,,,,,,,,,,,,,,,,,,,,,
0,1430198770,7,11,5,2098,1489,20,0,0,7,...,4,2,2,-52.0,2874,1,1796,0,51,0
1,1430220345,0,42,4,1188,1033,9,0,1,12,...,4,3,1,-5.0,2463,1,1974,0,63,1
2,1430227081,7,33,4,1319,1270,22,0,0,12,...,4,3,1,13.0,2130,0,0,1830,0,63


In [3]:
# Удаление признаков, отсутствующих в тестовой выборке (итоги матча)
target_col = 'radiant_win'
result_cols = ['duration', 'tower_status_radiant', 'tower_status_dire',
               'barracks_status_radiant', 'barracks_status_dire']

y = features[target_col]
X = features.drop(columns=[target_col] + result_cols)
print('Признаков после удаления:', X.shape[1])

Признаков после удаления: 102


### Пропуски в данных

In [4]:
# Подсчёт пропусков
missing = X.isnull().sum()
missing = missing[missing > 0]
print('Признаки с пропусками:')
print(missing)

Признаки с пропусками:
first_blood_time               19553
first_blood_team               19553
first_blood_player1            19553
first_blood_player2            43987
radiant_bottle_time            15691
radiant_courier_time             692
radiant_flying_courier_time    27479
radiant_first_ward_time         1836
dire_bottle_time               16143
dire_courier_time                676
dire_flying_courier_time       26098
dire_first_ward_time            1826
dtype: int64


> **Вопрос 1:** Признаки c пропусками перечислены выше. Под возможными причинами пропусков:
> - `first_blood_time`, `first_blood_team`, `first_blood_player1`, `first_blood_player2` – если первая кровь не пролилась за первые 5 минут, то событие отсутствует.
> - `radiant_bottle_time`, `radiant_courier_time`, `radiant_flying_courier_time`, `radiant_first_ward_time`, `dire_bottle_time` и аналогичные – когда команда не купила соответствующий предмет или не установила наблюдателя за 5 минут.
> 
> **Вопрос 2:** Целевая переменная: `radiant_win`.

In [5]:
# Замена пропусков нулями
X.fillna(0, inplace=True)
print('Пропусков осталось:', X.isnull().sum().sum())

Пропусков осталось: 0


## 1. Градиентный бустинг "в лоб"

Кросс‑валидация по 5 блокам с перемешиванием. Тестируем 10, 20 и 30 деревьев.

In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=228)

In [7]:
auc_gb = {}

for n in [10, 20, 30]:
    gb = GradientBoostingClassifier(n_estimators=n, random_state=228)
    start = datetime.datetime.now()
    scores = cross_val_score(gb, X, y, cv=kf, scoring='roc_auc')
    elapsed = datetime.datetime.now() - start
    auc_gb[n] = (scores.mean(), elapsed)
    print(f'n_estimators={n}: AUC={scores.mean():.4f}, время {elapsed}')

n_estimators=10: AUC=0.6647, время 0:00:32.265343
n_estimators=20: AUC=0.6823, время 0:01:03.314181
n_estimators=30: AUC=0.6893, время 0:01:34.628530


> **Вопрос 3:** При увеличении числа деревьев качество растёт по логарифму, при том что времязатраты растут линейно.
> 
> **Вопрос 4:** Использовать более 30 деревьев нецелесообразно. Ускорить обучение можно уменьшением глубины деревьев (`max_depth=2`) или использованием подвыборки (`subsample=0.5`).

## 2. Логистическая регрессия

### 2.1 На всех исходных признаках (с масштабированием)

In [8]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
best_auc_lr_full = 0
best_C_full = None

for C in C_values:
    lr = LogisticRegression(C=C, solver='lbfgs', max_iter=1000, random_state=228)
    scores = cross_val_score(lr, X_scaled, y, cv=kf, scoring='roc_auc')
    mean_auc = scores.mean()
    if mean_auc > best_auc_lr_full:
        best_auc_lr_full = mean_auc
        best_C_full = C
    print(f'C={C:.3f}: AUC={mean_auc:.4f}')

print(f'\nЛучшая C={best_C_full}, AUC={best_auc_lr_full:.4f}')

C=0.001: AUC=0.7163
C=0.010: AUC=0.7164
C=0.100: AUC=0.7164
C=1.000: AUC=0.7164
C=10.000: AUC=0.7164
C=100.000: AUC=0.7164
C=1000.000: AUC=0.7164

Лучшая C=0.01, AUC=0.7164


> **Вопрос 1:** Вообще, логистическая регрессия не улавливает нелинейные зависимости, поэтому её результат должен быть несколько хуже, чем градиентный бустинг, однако в данном случае мы видим улучшение качества.

### 2.2 После удаления категориальных признаков

Удалим столбцы `lobby_type`, `r1_hero`–`r5_hero`, `d1_hero`–`d5_hero` (их имена содержат `hero` или `lobby`).

In [9]:
hero_pattern = [f'{side}{p}_hero' for side in ['r','d'] for p in range(1,6)]
cat_cols = ['lobby_type'] + hero_pattern
X_numeric = X.drop(columns=cat_cols, errors='ignore')
print('Форма после удаления категориальных:', X_numeric.shape)

Форма после удаления категориальных: (97230, 91)


In [10]:
scaler_num = StandardScaler()
X_num_scaled = scaler_num.fit_transform(X_numeric)

best_auc_lr_num = 0
best_C_num = None

for C in C_values:
    lr = LogisticRegression(C=C, solver='lbfgs', max_iter=1000, random_state=228)
    scores = cross_val_score(lr, X_num_scaled, y, cv=kf, scoring='roc_auc')
    mean_auc = scores.mean()
    if mean_auc > best_auc_lr_num:
        best_auc_lr_num = mean_auc
        best_C_num = C
    print(f'C={C:.3f}: AUC={mean_auc:.4f}')

print(f'\nЛучшая C={best_C_num}, AUC={best_auc_lr_num:.4f}')

C=0.001: AUC=0.7163
C=0.010: AUC=0.7165
C=0.100: AUC=0.7164
C=1.000: AUC=0.7164
C=10.000: AUC=0.7164
C=100.000: AUC=0.7164
C=1000.000: AUC=0.7164

Лучшая C=0.01, AUC=0.7165


> **Вопрос 2:** После удаления категориальных признаков качество изменилось ну совсем чуть-чуть. Вероятно, это из-за того, что идентификаторы героев - не числовые величины, и их подача в линейную модель без дополнительной обработки будет вносить только шум.

### 2.3 Количество уникальных героев

In [11]:
hero_ids = pd.concat([features[col] for col in hero_pattern], ignore_index=True)
unique_heroes = hero_ids.dropna().unique()
N = len(unique_heroes)
print(f'Уникальных героев: {N}')

Уникальных героев: 108


> **Вопрос 3:** Число различных идентификаторов героев в данных равно что-то около 108.

### 2.4 «Мешок слов» по героям

In [12]:
# Создаём матрицу мешка слов
X_pick = np.zeros((features.shape[0], N+4))

for i, match_id in enumerate(features.index):
    for p in range(5):
        X_pick[i, features.loc[match_id, 'r%d_hero' % (p+1)]-1] = 1
        X_pick[i, features.loc[match_id, 'd%d_hero' % (p+1)]-1] = -1

# Объединяем с числовыми признаками
X_combined = np.hstack([X_numeric.values, X_pick])
print('Итоговая размерность:', X_combined.shape)

Итоговая размерность: (97230, 203)


In [13]:
scaler_comb = StandardScaler()
X_comb_scaled = scaler_comb.fit_transform(X_combined)

best_auc_lr_comb = 0
best_C_comb = None

for C in C_values:
    lr = LogisticRegression(C=C, solver='lbfgs', max_iter=1000, random_state=228)
    scores = cross_val_score(lr, X_comb_scaled, y, cv=kf, scoring='roc_auc')
    mean_auc = scores.mean()
    if mean_auc > best_auc_lr_comb:
        best_auc_lr_comb = mean_auc
        best_C_comb = C
    print(f'C={C:.3f}: AUC={mean_auc:.4f}')

print(f'\nЛучшая C={best_C_comb}, AUC={best_auc_lr_comb:.4f}')

C=0.001: AUC=0.7517
C=0.010: AUC=0.7520
C=0.100: AUC=0.7519
C=1.000: AUC=0.7519
C=10.000: AUC=0.7519
C=100.000: AUC=0.7519
C=1000.000: AUC=0.7519

Лучшая C=0.01, AUC=0.7520


> **Вопрос 4:** Добавление "мешка слов" немного улучшает качество логистической регрессии. Видимо модель учитывает индивидуальную силу героев и их командные комбинации.

# Делаем лучшую модель и предсказываем тесты
По кросс‑валидации выбираем модель с наивысшим AUC – логистическая регрессия с «мешком слов».

In [14]:
# Обучаем финальную модель на всей обучающей выборке
final_model = LogisticRegression(C=best_C_comb, solver='lbfgs', max_iter=1000, random_state=228)
final_model.fit(X_comb_scaled, y)
print('Модель обучена!')

Модель обучена!


### Загрузка и подготовка тестовой выборки

In [15]:
test = pd.read_csv('./data/features_test.csv', index_col='match_id')
print('Тестовых объектов:', test.shape[0])

# Всё то же: заполнение пропусков
test.fillna(0, inplace=True)

# Формируем мешок слов для героев
X_test_pick = np.zeros((test.shape[0], N+4))
for i, match_id in enumerate(test.index):
    for p in range(5):
        r_hero = test.loc[match_id, f'r{p+1}_hero']
        d_hero = test.loc[match_id, f'd{p+1}_hero']
        if pd.notna(r_hero):
            X_test_pick[i, int(r_hero) - 1] = 1
        if pd.notna(d_hero):
            X_test_pick[i, int(d_hero) - 1] = -1

# Удаляем hero/lobby из теста и объединяем
test_numeric = test.drop(columns=cat_cols, errors='ignore').values
X_test_combined = np.hstack([test_numeric, X_test_pick])

# Применяем сохранённый scaler
X_test_scaled = scaler_comb.transform(X_test_combined)

Тестовых объектов: 17177


In [16]:
# Предсказание вероятностей
test_proba = final_model.predict_proba(X_test_scaled)[:, 1]
print('Минимальная вероятность:', test_proba.min())
print('Максимальная вероятность:', test_proba.max())

Минимальная вероятность: 0.008505990698427108
Максимальная вероятность: 0.9962795503882279


> **Вопрос 5:** Минимальное и максимальное значения прогноза на тестовой выборке лежат в интервале $(0,1)$ и не совпадают, значит модель весьма адекватна.

In [17]:
# Сохранение результата
submission = pd.DataFrame({
    'match_id': test.index,
    'radiant_win': test_proba
})
submission.to_csv('submission.csv', index=False)
print('Файл submission.csv сохранён. Примеры строк:')
submission.head()

Файл submission.csv сохранён. Примеры строк:


,match_id,radiant_win
0,6,0.822791
1,7,0.752167
2,10,0.189067
3,13,0.857639
4,16,0.243964


# мы выполнили задание, ура